# 🏭 Hot Rolling Defect Detection — v2 (Calibrated Ensemble)

**The fix:** We use Stratified K-Fold to generate OOF probabilities, but instead of tuning the threshold on OOF scores directly, we **calibrate** each model's probabilities using `CalibratedClassifierCV`, making them meaningful and comparable. The threshold is then swept on OOF probabilities of the calibrated ensemble.

Place `train.csv` and `test.csv` in the same directory, then Run All.

In [1]:
!pip install lightgbm xgboost catboost scikit-learn pandas numpy --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import recall_score, precision_score, roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
print('Imports OK ✅')

Imports OK ✅


## Step 1 — Load Data

In [3]:
train = pd.read_csv(r'C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\train.csv')
test  = pd.read_csv(r'C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\test.csv')
feature_cols = [c for c in train.columns if c.startswith('X')]
print(f'Train: {train.shape} | Defects: {train["Y"].sum()} ({train["Y"].mean()*100:.1f}%)')
print(f'Test : {test.shape}')
print(f'Features: {len(feature_cols)}')

Train: (1352, 51) | Defects: 66.0 (4.9%)
Test : (339, 50)
Features: 49


## Step 2 — Feature Engineering

In [4]:
def engineer_features(df):
    df = df.copy()
    feat = df[feature_cols].values

    # Deltas: inter-stage differences
    for i in range(len(feature_cols) - 1):
        a, b = feature_cols[i], feature_cols[i+1]
        df[f'delta_{a}_{b}'] = df[b] - df[a]
    delta_cols = [c for c in df.columns if c.startswith('delta_')]

    # Row-wise stability
    df['mean_all']  = np.nanmean(feat, axis=1)
    df['std_all']   = np.nanstd(feat, axis=1)
    df['max_all']   = np.nanmax(feat, axis=1)
    df['min_all']   = np.nanmin(feat, axis=1)
    df['range_all'] = df['max_all'] - df['min_all']
    df['iqr_all']   = np.nanpercentile(feat, 75, axis=1) - np.nanpercentile(feat, 25, axis=1)
    row_mean = df['mean_all'].values[:, None]
    row_std  = np.where(df['std_all'].values[:, None] == 0, 1e-9, df['std_all'].values[:, None])
    df['n_outliers'] = (np.abs((feat - row_mean) / row_std) > 2).sum(axis=1)
    df['cv']         = df['std_all'] / (np.abs(df['mean_all']) + 1e-9)

    # Cumulative & rolling
    df['cumsum_std'] = np.std(np.nancumsum(feat, axis=1), axis=1)
    rolling_stds = np.stack([np.nanstd(feat[:, k-2:k+1], axis=1) for k in range(2, feat.shape[1])], axis=1)
    df['rolling3_std_mean'] = np.nanmean(rolling_stds, axis=1)
    df['rolling3_std_max']  = np.nanmax(rolling_stds, axis=1)

    # Delta stability
    dvals = df[delta_cols].values
    df['delta_std']   = np.nanstd(dvals, axis=1)
    df['delta_max']   = np.nanmax(np.abs(dvals), axis=1)
    df['delta_n_big'] = (np.abs(dvals) > np.nanstd(dvals, axis=1)[:, None] * 2).sum(axis=1)
    return df

train_eng = engineer_features(train)
test_eng  = engineer_features(test)
eng_cols  = [c for c in train_eng.columns if c not in ('CoilID', 'Y')]
print(f'Total features after engineering: {len(eng_cols)}')

Total features after engineering: 111


## Step 3 — Preprocessing

In [5]:
X_raw   = train_eng[eng_cols].copy()
y_train = train_eng['Y'].values
X_t_raw = test_eng[eng_cols].copy()

medians = X_raw.median()
X_raw   = X_raw.fillna(medians)
X_t_raw = X_t_raw.fillna(medians)

scaler  = RobustScaler()
X_train = scaler.fit_transform(X_raw)
X_test  = scaler.transform(X_t_raw)

pos = y_train.sum(); neg = len(y_train) - pos
ratio = neg / pos
print(f'Class ratio: {ratio:.1f}x  |  Defects: {pos}  Normal: {neg}')

Class ratio: 19.5x  |  Defects: 66.0  Normal: 1286.0


## Step 4 — OOF Probability Collection with Calibration

Key insight: We collect OOF probabilities from **isotonic calibration** applied within each fold. This makes the probability scores meaningful (well-separated between classes) even with a small fold size, fixing the 0.002-threshold collapse we saw before.

In [6]:
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Base estimators (uncalibrated) — these get wrapped in CalibratedClassifierCV per fold
def make_models(ratio):
    return {
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.02, max_depth=5, num_leaves=20,
            min_child_samples=3, subsample=0.8, colsample_bytree=0.7,
            scale_pos_weight=ratio, reg_alpha=0.1, reg_lambda=1.0,
            random_state=42, verbose=-1, n_jobs=-1),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=800, max_depth=4, learning_rate=0.01,
            scale_pos_weight=ratio, subsample=0.8, colsample_bytree=0.7,
            min_child_weight=3, gamma=1, reg_alpha=0.1, reg_lambda=1.0,
            eval_metric='logloss', random_state=42, verbosity=0,
            n_jobs=-1, use_label_encoder=False),
        'CatBoost': cb.CatBoostClassifier(
            iterations=800, learning_rate=0.02, depth=5,
            scale_pos_weight=ratio, l2_leaf_reg=3, random_seed=42, verbose=0),
        'RandomForest': RandomForestClassifier(
            n_estimators=500, class_weight={0:1, 1:30},
            min_samples_leaf=2, random_state=42, n_jobs=-1),
        'ExtraTrees': ExtraTreesClassifier(
            n_estimators=500, class_weight={0:1, 1:30},
            min_samples_leaf=2, random_state=42, n_jobs=-1),
    }

model_names = list(make_models(ratio).keys())
oof_proba  = {n: np.zeros(len(y_train)) for n in model_names}
test_proba = {n: np.zeros(len(X_test))  for n in model_names}

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X_train, y_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    print(f'Fold {fold+1}/5 | defects in val: {y_val.sum()}')

    models = make_models(ratio)
    for name, base_model in models.items():
        # Train base model on fold
        base_model.fit(X_tr, y_tr)
        # Calibrate on validation fold (isotonic regression maps scores → probabilities)
        cal = CalibratedClassifierCV(base_model, cv='prefit', method='isotonic')
        cal.fit(X_val, y_val)
        oof_proba[name][val_idx] = cal.predict_proba(X_val)[:, 1]
        # For test: average calibrated predictions across folds
        test_proba[name] += cal.predict_proba(X_test)[:, 1] / 5

print('\nOOF AUC per model:')
for name, p in oof_proba.items():
    print(f'  {name:15s}: {roc_auc_score(y_train, p):.4f}')

Fold 1/5 | defects in val: 13.0


InvalidParameterError: The 'cv' parameter of CalibratedClassifierCV must be an int in the range [2, inf), an object implementing 'split' and 'get_n_splits', an iterable or None. Got 'prefit' instead.

## Step 5 — Ensemble + Threshold Sweep on OOF

Now that probabilities are calibrated, the OOF sweep gives a **trustworthy** threshold estimate.

In [ ]:
oof_ens  = np.mean(list(oof_proba.values()), axis=0)
test_ens = np.mean(list(test_proba.values()), axis=0)

print('Calibrated OOF probability stats:')
print(f'  Defect (y=1) | min={oof_ens[y_train==1].min():.4f}  mean={oof_ens[y_train==1].mean():.4f}  max={oof_ens[y_train==1].max():.4f}')
print(f'  Normal (y=0) | min={oof_ens[y_train==0].min():.4f}  mean={oof_ens[y_train==0].mean():.4f}  max={oof_ens[y_train==0].max():.4f}')

thresholds = np.linspace(0.001, 0.99, 5000)
results = []
for t in thresholds:
    preds = (oof_ens >= t).astype(int)
    rec   = recall_score(y_train, preds, zero_division=0)
    prec  = precision_score(y_train, preds, zero_division=0)
    tp = int(((preds==1)&(y_train==1)).sum())
    fp = int(((preds==1)&(y_train==0)).sum())
    fn = int(((preds==0)&(y_train==1)).sum())
    results.append(dict(threshold=t, recall=rec, precision=prec, tp=tp, fp=fp, fn=fn, n_flagged=int(preds.sum())))

results_df = pd.DataFrame(results)

# Tier-1: perfect recall
candidates = results_df[results_df['recall'] == 1.0]
if len(candidates) == 0:
    print('\n⚠️  Relaxing to Recall ≥ 0.985 (at most 1 missed defect)')
    candidates = results_df[results_df['recall'] >= 0.985]
if len(candidates) == 0:
    print('⚠️  Using minimum-FN fallback')
    candidates = results_df[results_df['fn'] == results_df['fn'].min()]

best = candidates.sort_values('precision', ascending=False).iloc[0]
THRESHOLD = float(best['threshold'])

print(f'\n┌─────────────────────────────────┐')
print(f'│  Threshold : {THRESHOLD:.4f}')
print(f'│  OOF Recall    : {best["recall"]:.4f}')
print(f'│  OOF Precision : {best["precision"]:.4f}')
print(f'│  TP/FP/FN      : {best["tp"]}/{best["fp"]}/{best["fn"]}')
print(f'│  OOF flagged   : {best["n_flagged"]} / {len(y_train)}')
print(f'└─────────────────────────────────┘')

## Step 6 — Threshold Analysis Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(results_df['threshold'], results_df['recall'],    color='#e74c3c', lw=2, label='Recall')
ax.plot(results_df['threshold'], results_df['precision'], color='#2980b9', lw=2, label='Precision')
ax.axvline(THRESHOLD, color='#27ae60', ls='--', lw=1.8, label=f'Threshold={THRESHOLD:.3f}')
ax.axhline(1.0, color='#e74c3c', ls=':', alpha=0.3)
ax.axhline(0.9, color='#2980b9', ls=':', alpha=0.3)
ax.set(xlabel='Threshold', ylabel='Score', title='Recall & Precision vs Threshold (Calibrated OOF)', ylim=(0,1.05))
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(results_df['threshold'], results_df['n_flagged'], color='#8e44ad', lw=2)
ax2.axvline(THRESHOLD, color='#27ae60', ls='--', lw=1.8, label=f'Threshold={THRESHOLD:.3f}')
ax2.axhline(y_train.sum(), color='gray', ls=':', alpha=0.5, label=f'True defects={y_train.sum()}')
ax2.set(xlabel='Threshold', ylabel='Samples Flagged', title='Flagged Count vs Threshold (Calibrated OOF)')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Final Predictions on Test Set

In [ ]:
final_preds = (test_ens >= THRESHOLD).astype(int)
print(f'Threshold applied : {THRESHOLD:.4f}')
print(f'Test defects flagged: {final_preds.sum()} / {len(final_preds)}')
print(f'(Winner flagged 26 — if your number is very different, check AUC scores above)')

print('\nPer-model test flags at this threshold:')
for name, p in test_proba.items():
    print(f'  {name:15s}: {(p >= THRESHOLD).sum()}')

## Step 8 — Export Submission

In [ ]:
submission = pd.DataFrame({'CoilID': test['CoilID'].values, 'Y': final_preds})
submission.to_csv('expected_submission.csv', index=False)
print(f'Saved: expected_submission.csv  ({submission.shape[0]} rows × 2 cols)')
print(submission['Y'].value_counts().to_string())
submission.head(10)

---
## ✅ Done! Submit `expected_submission.csv` 🏭